In [1]:
import numpy as np
import ipytest
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from typing import Tuple, List, Dict, Any, Union

ipytest.autoconfig()

In [2]:
test_X = np.array([[1, 2, 3],
				   [3, 4, 5],
				   [5, 6, 7],
			       [7, 8, 9]])
test_y = np.array([1, 0, 1, 0])

计算基尼系数:
$$Gini(p) = 1 - \sum_{k=1}^{K} p_k^2$$

1. 首先统计 y 中各类别的数量。
2. 为了得到 $p_k$ 值，需要每个类别分别占多少类别。即用上面的数量除以总数得到每个类别的占比，这里用了向量的的表示方法。
3. 根据公式进行拼装。


In [3]:
def calculate_gini(y: np.ndarray) -> float:
    """计算数据集的基尼不纯度 (Gini Impurity)

    Args:
        y (np.ndarray): 样本的标签数组，形状为 (n_samples,)

    Returns:
        float: 当前标签集合的基尼不纯度
    """
    # 1. 统计 y 中各类别的数量 (可以使用 np.bincount 或 np.unique)
    count = np.bincount(y.astype(int))
    # 2. 计算每个类别的概率 p_k，得到一个向量
    probs = count / len(y)
    # 3. 根据公式 1 - sum(p_k^2) 计算并返回基尼不纯度
    return 1 - np.sum(probs ** 2)

这里测试单元建立了一个示例，也就是3个1，5个0组成的两类，对应笔记中[[基尼系数#1. 算“一个小团体”的纯度]]的式子


$$1 - [(\frac{3}{8})^2 + (\frac{5}{8})^2] = \frac{15}{32}=0.46875$$

In [4]:
def test_calculate_gini():
    # 3个1，5个0组成的y
    test_all_y = np.array([1, 1, 1, 0, 0, 0, 0, 0])
    gini = calculate_gini(test_all_y)
    assert gini == 0.46875

为了确认切分的情况，这里计算了基尼不纯度，它的值越低，效果越好。

$$Gini(D, A) = \frac{|D_1|}{|D|} Gini(D_1) + \frac{|D_2|}{|D|} Gini(D_2)$$

In [5]:
def calculate_gini_impurity(y: np.ndarray, y1: np.ndarray, y2: np.ndarray) -> float:
    """计算切分后的加权基尼不纯度

    Args:
        y (np.ndarray): 切分前的标签
        y1 (np.ndarray): 切分后左子树的标签
        y2 (np.ndarray): 切分后右子树的标签

    Returns:
        float: 加权基尼不纯度
    """

    left_weight = len(y1)/len(y)
    right_weight = len(y2)/len(y)
    left_gini = calculate_gini(y1)
    right_gini = calculate_gini(y2)
    return left_weight * left_gini + right_weight * right_gini

这里测试单元建立了一个示例，同样对应笔记中[[基尼系数#2. 计算“整个特征”的好坏]]的式子


$$\underbrace{\frac{8}{15}}_{\frac{|D_1|}{|D|}} \times \underbrace{\frac{15}{32}}_{Gini(D_1)} + \underbrace{\frac{7}{15}}_{\frac{|D_2|}{|D|}} \times \underbrace{\frac{20}{49}}_{Gini(D_2)} = \frac{37}{84} \approx 0.4405$$

- 总人数为15
- 男性为8人
- 女性为7人
- 男性的基尼系数为$\frac{15}{32}$
- 女性的基尼系数为$\frac{20}{49}$

In [6]:
def test_calculate_gini_impurity():
    man_y = np.array([1, 1, 1, 0, 0, 0, 0, 0])
    woman_y = np.array([1, 1, 0, 0, 0, 0, 0])
    all_y = np.hstack((man_y, woman_y))
    gini_impurity = calculate_gini_impurity(all_y, man_y, woman_y)
    assert round(gini_impurity, 4) == 0.4405

In [7]:
def split_dataset(X: np.ndarray, y: np.ndarray, feature_idx: int, threshold: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """根据给定的特征索引和阈值，将数据集一分为二 (左子树和右子树)

    Args:
        X (np.ndarray): 训练样本输入
        y (np.ndarray): 训练样本标签
        feature_idx (int): 用于切分的特征列索引
        threshold (float): 用于切分的阈值

    Returns:
        Tuple: (X_left, y_left, X_right, y_right)
    """
    # 1. 找到 X[:, feature_idx] <= threshold 的样本索引作为左子集
    # 2. 找到 X[:, feature_idx] > threshold 的样本索引作为右子集
    mask = X[:, feature_idx] <= threshold
    # 3. 根据索引切分出 X_left, y_left 和 X_right, y_right 并返回
    X_left = X[mask]
    X_right = X[~mask]
    y_left = y[mask]
    y_right = y[~mask]

    return {
        "X_left":X_left, "y_left":y_left,
        "X_right":X_right, "y_right":y_right
    }

In [8]:
def test_split_dataset():
    split_data = split_dataset(test_X, test_y, 1, 5)
    assert np.array_equal(split_data["X_left"], np.array([[1, 2, 3], [3, 4, 5]]))
    assert np.array_equal(split_data["y_left"], np.array([1, 0]))
    assert np.array_equal(split_data["X_right"], np.array([[5, 6, 7], [7, 8, 9]]))
    assert np.array_equal(split_data["y_right"], np.array([1, 0]))

In [9]:
def find_best_split(X: np.ndarray, y: np.ndarray) -> Dict[str, Any]:
    """遍历所有特征和可能的阈值，寻找使基尼不纯度下降最多的最优切分点

    Args:
        X (np.ndarray): 训练样本输入
        y (np.ndarray): 训练样本标签

    Returns:
        Dict: 包含最优切分信息的字典，如 {'feature_idx': int, 'threshold': float, 'X_left': ..., 'y_left': ..., 'X_right': ..., 'y_right': ...}
              如果无法切分（例如纯度已经最高），返回空字典 {}
    """
    # 1. 初始化最小基尼不纯度 min_gini 为无穷大，以及一个空的 best_split 字典
    min_gini = float("inf")
    best_split = None

    # 2. 遍历每一个特征列 (feature_idx)
    for feature_idx in range(X.shape[1]):
        # 3. 对当前特征列的每一个唯一值 (可以使用 np.unique 提取) 作为候选阈值 threshold
        thresholds = np.unique(X[:, feature_idx])
        for threshold in thresholds:
            # 4. 调用 split_dataset 尝试切分数据集
            split_data = split_dataset(X, y, feature_idx, threshold)
            # 5. 调用 calculate_gini_impurity 计算切分后的加权基尼不纯度
            gini = calculate_gini_impurity(y, split_data["y_left"], split_data["y_right"])

            # 6. 如果加权基尼不纯度小于 min_gini，则更新 min_gini 和 best_split 字典的内容
            if gini < min_gini:
                min_gini = gini
                best_split = split_data
                best_split['feature_idx'] = feature_idx
                best_split['threshold'] = threshold

    # 7. 循环结束后返回 best_split
    return best_split

In [10]:
def leaf_value_calculation(y: np.ndarray) -> Any:
    """多数投票（定义叶子结点的值）
       返回 y 中出现次数最多的类别

    Args:
        y (np.ndarray): 当前的标签

    Returns:
        Any: 得票最多的类型的值
    """

    y_flat = y.flatten().astype(int)
    # 统计每个非负整数出现的次数，并返回出现次数最多的索引（即对应的值）
    return np.bincount(y_flat).argmax()

In [11]:
def build_tree(X: np.ndarray, y: np.ndarray, current_depth: int = 0, max_depth: int = 3, min_samples_split: int = 2) -> Dict[str, Any]:
    """递归构建 CART 决策树

    Args:
        X (np.ndarray): 训练样本输入
        y (np.ndarray): 训练样本标签
        current_depth (int): 当前树的深度
        max_depth (int): 树的最大允许深度
        min_samples_split (int): 节点分裂所需的最小样本数

    Returns:
        Dict: 嵌套字典形式的树结构。如果是叶子节点，返回 {'value': 预测的类别}；
              如果是内部节点，返回 {'feature_idx': ..., 'threshold': ..., 'left': 递归子树, 'right': 递归子树}
    """
    # 1. 检查停止条件：
    #          - 样本数少于 min_samples_split
    #          - 当前深度达到 max_depth
    #          - y 中的样本全部属于同一个类别 (纯度极高)
    if X.shape[0] < min_samples_split or \
       current_depth == max_depth or \
       len(np.unique(y)) == 1:
        # 2. 如果满足上述任一条件，返回叶子节点：求出 y 中出现次数最多的类别作为 'value'
        return {
            "is_leaf": True,
            "value": leaf_value_calculation(y)
        }
    # 3. 否则，调用 find_best_split 寻找最优切分点
    best_split = find_best_split(X, y)
    # 4. 如果 find_best_split 没有找到有效的切分（返回空字典），同时检查有没有一侧为空。也作为叶子节点返回
    if best_split is None or \
       len(best_split['y_left']) == 0 or \
       len(best_split["y_right"]) == 0:
        return {
            "is_leaf": True,
            "value": leaf_value_calculation(y)
        }
    # 5. 递归调用 build_tree 构建左子树 (传入 X_left, y_left) 和右子树 (传入 X_right, y_right)，深度 current_depth + 1
    left_node = build_tree(best_split['X_left'], best_split['y_left'], current_depth + 1, max_depth, min_samples_split)
    right_node = build_tree(best_split['X_right'], best_split['y_right'], current_depth + 1, max_depth, min_samples_split)

    # 6. 返回包含 'feature_idx', 'threshold', 'left', 'right' 的字典节点
    return {
        "is_leaf": False,
        "feature_idx": best_split["feature_idx"],
        "threshold": best_split["threshold"],
        "left_node": left_node,
        "right_node": right_node,
    }

In [12]:
def predict_single_sample(tree: Dict[str, Any], x: np.ndarray) -> float:
    """使用训练好的单棵决策树，对单个样本进行预测

    Args:
        tree (Dict): build_tree 返回的嵌套字典
        x (np.ndarray): 单个样本的特征向量 (1 维数组)

    Returns:
        float: 该样本落入的叶子结点的预测 (value)
    """

    # 1. 检查当前结点 tree["is_leaf"] 是否为 True。
    if tree['is_leaf']:
        return tree['value']
    # 2. 判断该样本的对应特征值 x[feature_idx] 是否 <= threshold。
    if x[tree["feature_idx"]] <= tree["threshold"]:
        # 4. 如果是，递归调用 predict_single_sample，传入 tree["left_node"] 和 x。
        value = predict_single_sample(tree['left_node'], x)
        # 5. 如果不是，递归调用 predict_single_sample，传入 tree["right_node"] 和 x。
    else:
        value = predict_single_sample(tree['right_node'], x)

    return value

In [13]:
def predict_tree(tree: Dict[str, Any], X: np.ndarray) -> np.ndarray:
    """使用训练好的递归树字典对新数据进行预测

    Args:
        tree (Dict): 通过 build_tree 构建的嵌套字典树结构
        X (np.ndarray): 测试样本输入，形状为 (n_samples, n_features)

    Returns:
        np.ndarray: 预测的标签数组
    """

    # 1. 对 X 中的每一行数据应用 `predict_single_sample` (可以用列表推导式)
    y_perd = [predict_single_sample(tree, x) for x in X]
    # 2. 将结果转换为 np.ndarray 返回
    return np.array(y_perd)

随机森林的精髓。

先把样本摇匀了，然后再抽样的时候再摇一次。

In [14]:
def bootstrap_sampling(X: np.ndarray, y: np.ndarray, n_estimators: int, rng: np.random.Generator) -> List[Tuple[np.ndarray, np.ndarray]]:
    """自助抽样选择训练数据子集

    Args:
        X (np.ndarray): 训练样本输入，形状为 (n_samples, n_features)
        y (np.ndarray): 训练样本标签，形状为 (n_samples,)
        n_estimators (int): 树的棵数
        rng (np.random.Generator): 随机化的类实现

    Returns:
        List[Tuple[np.ndarray, np.ndarray]]: 抽样子集列表，每个元素包含一个元组 (bootstrap_X, bootstrap_y)
    """
    # 初始化np的随机对象，初始化用于返回的打乱后的数据集列表
    bootstrap_dataset = []
    # 1. 将 X 和 y 按列合并以便同时打乱
    dataset = np.hstack((X, y.reshape(-1, 1)))
    # 2. 打乱合并后的数据集
    rng.shuffle(dataset)
    # 3. 循环 n_estimators 次，每次利用 np.random.choice 进行有放回的行抽样 (第一个随机性)
    for _ in range(n_estimators):
        current_dataset = rng.choice(dataset, dataset.shape[0], axis=0, replace=False)
        # 4. 将抽样得到的特征和标签分别提取出来，存入列表并返回
        bootstrap_X = current_dataset[:, :-1]
        bootstrap_y = current_dataset[:, -1]
        bootstrap_dataset.append((bootstrap_X, bootstrap_y))

    return bootstrap_dataset

In [15]:
def test_bootstrap_sampling():
    rng = np.random.default_rng(seed=2)
    bootstrap_dataset = bootstrap_sampling(test_X, test_y, 3, rng)
    assert len(bootstrap_dataset) == 3
    assert bootstrap_dataset[0][0].shape == (4, 3)
    assert bootstrap_dataset[0][1].shape == (4,)
    assert np.array_equal(bootstrap_dataset[0][0],
                          np.array([[3, 4, 5], [1, 2, 3], [7, 8, 9],[5, 6, 7]]))

模型训练返回的值如下：

```mermaid
mindmap
)随机森林模型(
  ((模型列表<br> List))
    feature_index<br>索引特征，对应树中每次随机抽取的特征索引<br>numpy.ndarray ﬂ°°40¶ß1,ﬂ°°41¶ß
    (tree<br>树<br> Dict)
      内部结点
        feature_idx<br>决定当前岔路根据哪一列特征进行判断。<br>必须 | Integer
        threshold<br>决定当前路口的阈值，例如：当前特征是否小于等于 2.45。<br> 必须 | Float
        left and right<br>指向下一个树的子结点指针，嵌套的字典结构<br>必须 | Dict
      叶子结点
        value</br>该结点的最终预测输出得分，即 y 中出现次数最多的类别</br>必须 | int
        is_leaf<br>标志位，直接获取可以判断是否是叶子节点<br>必须 | Boolean
```

In [16]:
def fit_random_forest(X: np.ndarray,
                      y: np.ndarray,
                      n_estimators: int,
                      rng: np.random.Generator,
                      max_features: int = None) -> List[Any]:
    """定义随机森林训练方法

    Args:
        X (np.ndarray): 训练样本输入
        y (np.ndarray): 训练样本标签
        n_estimators (int): 需要训练的树的棵数
        rng (np.random.Generator): 随机化的类实现
        max_features (int, optional): 每次分裂随机选择的最大特征数。默认为 None (通常可设为特征数的平方根)

    Returns:
        List[Any]: 训练好的决策树对象列表（每棵树需要附带其在训练时被分配的特征索引）
    """
    model = []
    # 1. 调用 bootstrap_sampling 获取各个树的训练子集
    bootstrap_dataset = bootstrap_sampling(X, y, n_estimators, rng)
    # 2. 确定 max_features 的大小（如果没有传入，建议取总特征数的平方根）
    max_features = int(np.sqrt(X.shape[0])) if max_features is None else max_features
    # 3. 循环初始化 n_estimators 棵基础分类树
    for i in range(n_estimators):
        # 4. 对每一棵树，利用 np.random.choice 从所有特征中无放回地随机抽取 max_features 个特征列 (第二个随机性)
        feather_index = rng.choice(bootstrap_dataset[i][0].shape[1], max_features, replace=False)
        bootstrap_X = bootstrap_dataset[i][0][:, feather_index]
        bootstrap_y = bootstrap_dataset[i][1]
        # 5. 使用抽样后的行和列数据去拟合 (fit) 对应的决策树
        tree = build_tree(bootstrap_X, bootstrap_y)
        model.append({
            "tree":tree,
            "feather_index":feather_index,
        })
    # 6. 返回树模型
    return model

In [17]:
def predict_random_forest(X: np.ndarray, model: List[Any]) -> List[int]:
    """随机森林预测

    Args:
        X (np.ndarray): 测试样本输入，形状为 (n_samples, n_features)
        trees (List[Any]): 已经训练好的决策树列表（包含每棵树的 feature_indices 属性）

    Returns:
        List[int]: 最终集成的分类预测结果列表
    """
    # 1. 初始化一个列表用于收集每棵树的预测结果
    y_preds = []

    # 2. 遍历 model 列表，读取每棵树专属的特征列索引
    for m in model:
        # 3. 根据特征索引从 X 中截取对应的子特征输入，获取该树的预测结果并保存
        y_pred = predict_tree(m['tree'], X[:, m['feather_index']])
        y_preds.append(y_pred)

    # 4. 将所有树的预测结果矩阵进行转置，使得每一行代表一个样本的所有树预测值
    y_preds = np.array(y_preds).T
    # 5. 对每个样本的所有预测结果使用“投票法”（如利用 np.bincount 找出现次数最多的类别）得出最终结果并返回
    temp = np.where(y_preds == 0, -1, 1)
    temp = np.sum(temp, axis=1)
    y_preds = np.where(temp >= 0, 1, 0)
    return y_preds


In [18]:
# 树的棵数
n_estimators = 10
# 列抽样最大特征数
max_features = 15
# 生成模拟二分类数据集
X, y = make_classification(n_samples=1000, n_features=20, n_redundant=0, n_informative=2,
                           random_state=1, n_clusters_per_class=1)
rng = np.random.default_rng(seed=2)
X += 2 * rng.uniform(size=X.shape)
# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [19]:
ipytest.run("-qq")

....                                                                                         [100%]


<ExitCode.OK: 0>

In [20]:
print(f'X_train: {X_train.shape}\n' \
      f'y_train: {y_train.shape}\n' \
      f'X_test: {X_test.shape}\n' \
      f'y_test: {y_test.shape}')

X_train: (700, 20)
y_train: (700,)
X_test: (300, 20)
y_test: (300,)


In [21]:
model = fit_random_forest(X_train, y_train, n_estimators, rng, max_features)

In [22]:
y_pred = predict_random_forest(X_test, model)

In [23]:
# 准确率评估
accuracy = accuracy_score(y_test, y_pred)
print ("Accuracy: ", accuracy)

Accuracy:  0.8566666666666667
